## Cell 1 - imports

In [11]:
# Import the libraries used in this notebook section.
import json
import os
import re
import time
import pandas as pd
from deep_translator import GoogleTranslator
from tqdm import tqdm

## Cell 2 - load cleaned dataset

In [12]:
cleaned_file = "../data/processed/ted_cleaned_for_cost_prediction.csv"
# Load the dataset needed for the next analysis step.
df_clean = pd.read_csv(cleaned_file, low_memory=False)

# Inspect the data to confirm the structure and values look reasonable.
print("Shape:", df_clean.shape)
display(df_clean.head())

Shape: (46503, 15)


,TITLE,CPV,ADDITIONAL_CPVS,TYPE_OF_CONTRACT,TOP_TYPE,MAIN_ACTIVITY,CAE_TYPE,ISO_COUNTRY_CODE,TAL_LOCATION_NUTS,LOTS_NUMBER,VALUE_EURO,DISPATCH_YEAR,DISPATCH_MONTH,DISPATCH_QUARTER,LOG_VALUE_EURO
0,Unknown,85000000,Unknown,S,NIC,Health,6,AT,AT,1.0,16257000.00,2022,12,4,16.604034
1,Suministro de diversos vehículos para el Parqu...,34144900,Unknown,U,OPE,General public\services,3,ES,ES521,1.0,109000.00,2022,12,4,11.599112
2,Laboratórne zariadenia,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272
3,Zariadenia na spracovanie vzoriek,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272
4,Zariadenia pre potreby experimentálneho zverinca,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272


## Cell 3 - check TITLE column exists

In [13]:
if "TITLE" not in df_clean.columns:
    raise ValueError("The cleaned dataset does not contain a 'TITLE' column.")

print("TITLE column found.")
# Handle missing values so later transformations and models do not fail.
print("Non-null TITLE rows:", df_clean["TITLE"].notna().sum())

TITLE column found.
Non-null TITLE rows: 46503


## Cell 4 - translation settings

In [14]:
# Safe defaults:
# - FAST_SKIP_ENGLISH=True skips titles that already look clearly English
# - checkpointing means you can rerun without losing progress
FAST_SKIP_ENGLISH = True
BATCH_SIZE = 60
MAX_BATCH_CHARS = 3000
ENABLE_CHECKPOINTS = False
PAUSE_SECONDS = 0.0

print({
    "FAST_SKIP_ENGLISH": FAST_SKIP_ENGLISH,
    "BATCH_SIZE": BATCH_SIZE,
    "MAX_BATCH_CHARS": MAX_BATCH_CHARS,
    "ENABLE_CHECKPOINTS": ENABLE_CHECKPOINTS,
    "PAUSE_SECONDS": PAUSE_SECONDS
})

{'FAST_SKIP_ENGLISH': True, 'BATCH_SIZE': 60, 'MAX_BATCH_CHARS': 3000, 'ENABLE_CHECKPOINTS': False, 'PAUSE_SECONDS': 0.0}


## Cell 5 - normalise titles and identify what actually needs translation

In [15]:
def normalize_title(text):
    # Handle missing values so later transformations and models do not fail.
    if pd.isna(text):
        return pd.NA
    text = str(text).replace("\xa0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text

def should_translate(text):
    if pd.isna(text):
        return False
    text = str(text).strip()
    if text == "":
        return False
    if text.lower() in {"unknown", "n/a", "na", "none", "null"}:
        return False
    if len(text) < 4:
        return False
    if re.fullmatch(r"[\W\d_]+", text):
        return False
    return True

ENGLISH_HINT_WORDS = {
    "the", "and", "for", "of", "in", "to", "with", "without", "at", "on", "by",
    "contract", "contracts", "framework", "agreement", "services", "service",
    "supply", "supplies", "works", "work", "construction", "design", "maintenance",
    "repair", "rehabilitation", "installation", "equipment", "purchase", "procurement",
    "school", "hospital", "road", "roads", "housing", "building", "public", "cleaning",
    "security", "management", "consultancy", "consultant", "office", "waste", "water",
    "electricity", "it", "software", "hardware", "medical", "transport", "project"
}

def is_probably_english(text):
    if pd.isna(text):
        return False
    text = str(text).strip()

    # If the title contains non-ASCII characters, do not assume English
    if re.search(r"[^\x00-\x7F]", text):
        return False

    # Filter the data to keep the records relevant for this step.
    words = re.findall(r"[A-Za-z]+", text.lower())
    if not words:
        return False

    hits = sum(word in ENGLISH_HINT_WORDS for word in words)
    ratio = hits / len(words)

    # Very conservative: only skip when it looks clearly English
    if len(words) <= 3:
        return hits == len(words)
    return ratio >= 0.40

df_clean["TITLE_NORM"] = df_clean["TITLE"].map(normalize_title)

unique_titles = (
    df_clean["TITLE_NORM"]
    .dropna()
    .loc[lambda s: s.map(should_translate)]
    .drop_duplicates()
    .tolist()
)

if FAST_SKIP_ENGLISH:
    titles_to_translate = [t for t in unique_titles if not is_probably_english(t)]
    english_like_titles = [t for t in unique_titles if is_probably_english(t)]
else:
    titles_to_translate = unique_titles
    english_like_titles = []

print("Rows in dataset:", len(df_clean))
print("Unique non-empty candidate titles:", len(unique_titles))
print("Likely English titles kept as-is:", len(english_like_titles))
print("Unique titles sent for translation:", len(titles_to_translate))

# Inspect the data to confirm the structure and values look reasonable.
display(pd.DataFrame({
    "Category": [
        "Unique candidate titles",
        "Likely English skipped",
        "Titles to translate"
    ],
    "Count": [
        len(unique_titles),
        len(english_like_titles),
        len(titles_to_translate)
    ]
}))

Rows in dataset: 46503
Unique non-empty candidate titles: 25364
Likely English titles kept as-is: 125
Unique titles sent for translation: 25239


,Category,Count
0,Unique candidate titles,25364
1,Likely English skipped,125
2,Titles to translate,25239


## Cell 6 - resume from checkpoint if available

In [16]:
translation_map = {}

# Filter the data to keep the records relevant for this step.
titles_pending = [title for title in titles_to_translate if title not in translation_map]
print(f"Titles still pending: {len(titles_pending):,}")

print("Translation map will be kept in memory and used to build the final English dataset only.")


Titles still pending: 25,239
Translation map will be kept in memory and used to build the final English dataset only.


## Cell 7 - build translation batches

In [17]:
def make_batches(texts, batch_size=BATCH_SIZE, max_batch_chars=MAX_BATCH_CHARS):
    # Filter the data to keep the records relevant for this step.
    batches = []
    current_batch = []
    current_chars = 0

    for text in texts:
        text_len = len(text)

        if current_batch and (
            len(current_batch) >= batch_size or
            current_chars + text_len > max_batch_chars
        ):
            batches.append(current_batch)
            current_batch = []
            current_chars = 0

        current_batch.append(text)
        current_chars += text_len

    if current_batch:
        batches.append(current_batch)

    return batches

batches = make_batches(titles_pending)

print("Number of batches to process:", len(batches))
if batches:
    print("First batch size:", len(batches[0]))
    print("First batch preview:", batches[0][:5])

Number of batches to process: 571
First batch size: 53
First batch preview: ['Suministro de diversos vehículos para el Parque Móvil de la Diputación Provincial de Alicante - L2: “Suministro de dos furgones eléctricos con rampa interior plegable”', 'Laboratórne zariadenia', 'Zariadenia na spracovanie vzoriek', 'Zariadenia pre potreby experimentálneho zverinca', 'Upgrade existujúceho vybavenia']


## Cell 8 - translate in batches with fallback and checkpointing

In [18]:
translator = GoogleTranslator(source="auto", target="en")

for batch_idx, batch in enumerate(tqdm(batches, desc="Translating title batches"), start=1):
    try:
        translated_batch = translator.translate_batch(batch)

        if translated_batch is None or len(translated_batch) != len(batch):
            raise ValueError("Batch translation returned an unexpected result.")

        for source_text, translated_text in zip(batch, translated_batch):
            if isinstance(translated_text, str) and translated_text.strip():
                translation_map[source_text] = translated_text.strip()
            else:
                translation_map[source_text] = source_text

    except Exception:
        for source_text in batch:
            try:
                translated_text = translator.translate(source_text)
                if isinstance(translated_text, str) and translated_text.strip():
                    translation_map[source_text] = translated_text.strip()
                else:
                    translation_map[source_text] = source_text
            except Exception:
                translation_map[source_text] = source_text

    if ENABLE_CHECKPOINTS and batch_idx == len(batches):
        pass

    if PAUSE_SECONDS > 0:
        time.sleep(PAUSE_SECONDS)

print(f"Finished. Translation map now contains {len(translation_map):,} titles.")

Translating title batches: 100%|██████████| 571/571 [5:03:32<00:00, 31.90s/it]  

Finished. Translation map now contains 25,239 titles.


## Cell 9 - write translated titles back into TITLE

In [19]:
# Filter the data to keep the records relevant for this step.
df_clean["TITLE"] = df_clean["TITLE_NORM"]

translated_mask = df_clean["TITLE_NORM"].isin(translation_map.keys())
df_clean.loc[translated_mask, "TITLE"] = df_clean.loc[translated_mask, "TITLE_NORM"].map(translation_map)

# Preserve missing values
df_clean.loc[df_clean["TITLE_NORM"].isna(), "TITLE"] = pd.NA

# Inspect the data to confirm the structure and values look reasonable.
display(df_clean[["TITLE"]].head(20))

,TITLE
0,Unknown
1,Supply of various vehicles for the Mobile Park...
2,Laboratory equipment
3,Equipment for sample processing
4,Equipment for the needs of the experimental me...
5,Upgrade of existing equipment
6,Unknown
7,Unknown
8,Unknown
9,Unknown


## Cell 10 - final spot check

In [20]:
sample_check = (
    df_clean[["TITLE"]]
    .dropna()
    .drop_duplicates()
    .sample(min(20, len(df_clean[["TITLE"]].dropna().drop_duplicates())), random_state=42)
)

# Inspect the data to confirm the structure and values look reasonable.
display(sample_check)

,TITLE
7188,SUB Delivery of disinfectants
45350,HANGING WASTE BASKET - 200 PCS - II
45065,"MOP ""Górki Wschód"" located in the S61 expressw..."
29706,Employee gifts (subset 2)
20445,Painkillers
33239,Ballast and Aggregates: 32 lots
45379,Property protection and event services
7495,"COLLECTION, TRANSPORT AND DISPOSAL OF MEDICAL ..."
19660,"Degree Wodny Smolice, Ekologiczna 3, 32-640 Po..."
15486,Paris North Territorial Directorate


## Cell 11 - save translated dataset

In [21]:
output_file_final = "../data/processed/ted_cleaned_for_cost_prediction_english.csv"

# Filter the data to keep the records relevant for this step.
df_final = df_clean.drop(columns=["TITLE_NORM"], errors="ignore")
# Save the processed output so later notebooks or report sections can reuse it.
df_final.to_csv(output_file_final, index=False)

print(f"Saved new CSV to: {output_file_final}")
# Inspect the data to confirm the structure and values look reasonable.
display(df_final.head())

Saved new CSV to: ../data/processed/ted_cleaned_for_cost_prediction_english.csv


,TITLE,CPV,ADDITIONAL_CPVS,TYPE_OF_CONTRACT,TOP_TYPE,MAIN_ACTIVITY,CAE_TYPE,ISO_COUNTRY_CODE,TAL_LOCATION_NUTS,LOTS_NUMBER,VALUE_EURO,DISPATCH_YEAR,DISPATCH_MONTH,DISPATCH_QUARTER,LOG_VALUE_EURO
0,Unknown,85000000,Unknown,S,NIC,Health,6,AT,AT,1.0,16257000.00,2022,12,4,16.604034
1,Supply of various vehicles for the Mobile Park...,34144900,Unknown,U,OPE,General public\services,3,ES,ES521,1.0,109000.00,2022,12,4,11.599112
2,Laboratory equipment,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272
3,Equipment for sample processing,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272
4,Equipment for the needs of the experimental me...,38000000,42900000,U,OPE,Health,6,SK,SK,4.0,1942083.33,2022,12,4,14.479272


## Cell 12 - translation complete

In [22]:
print("Translation complete.")

Translation complete.
